# The `with` statement and context managers

This module covers Python's `with` statement and the context manager protocol. Readers are assumed to be comfortable with Python syntax and at least passing familiarity with TM1 cubes and the tm1py library; the focus here is on what context managers do, how to write them, and the situations in tm1py work where they are the right tool.

The mechanism exists to bracket a block of code with paired setup and teardown that always runs, even when the block raises. The classic example in tm1py is the `TM1Service` connection: it holds an HTTP session and a TM1 server login that must be released. The same machinery covers file handles, locks, sandbox switches, transactions, and timing blocks.

The topics below are arranged linearly for review. Structural grouping (sections, chapters) can be applied later.

---

## Topic list

1. The setup and teardown problem
2. The `with` statement
3. The context manager protocol
4. Writing a context manager as a class
5. Returning a value from `__enter__`
6. Handling exceptions in `__exit__`
7. The `contextlib.contextmanager` decorator
8. Multiple context managers in one statement
9. `ExitStack` for dynamic resources
10. Asynchronous context managers
11. Real-world design principles
12. Common mistakes

---

## 1. The setup and teardown problem

Many resources require paired operations: open a file and close it, acquire a lock and release it, log into TM1 and log out. The pairing must hold under exceptions. Without `with`, the only correct pattern is `try`/`finally`:

In [ ]:
from TM1py import TM1Service

tm1 = TM1Service(address="tm1.example.com", port=8001, user="admin", password="apple", ssl=True)
try:
    cube_names = tm1.cubes.get_all_names()
    # ... work with the connection ...
finally:
    tm1.logout()

The teardown step is several lines away from the setup, easy to forget, and easy to drop during refactoring. If a third operation is added between setup and the `try`, the cleanup is no longer guaranteed. Each new resource doubles the indentation. The `with` statement folds this pattern into a single line and attaches the cleanup to the resource itself rather than to its caller.

## 2. The `with` statement

A `with` block names a context manager and binds the result of entering it to a local variable. When the block exits, by any path, the manager's teardown runs:

In [ ]:
from TM1py import TM1Service

with TM1Service(
    address="tm1.example.com", port=8001,
    user="admin", password="apple", ssl=True,
) as tm1:
    cube_names = tm1.cubes.get_all_names()
    # cube_names: list[str], for example ["Sales Plan", "General Ledger", ...]

The block exits cleanly on `return`, on falling off the end, on `break` or `continue` from an enclosing loop, and on an unhandled exception. In every case `tm1.logout()` runs before control leaves the block. The same construct works for files (`open(...) as f`), locks (`threading.Lock() as lock`), and any object that implements the protocol described next.

## 3. The context manager protocol

A context manager is any object that implements two methods: `__enter__(self)` and `__exit__(self, exc_type, exc_value, traceback)`. The `with` statement is roughly equivalent to:

In [ ]:
manager = TM1Service(...)
value = manager.__enter__()
try:
    # body of the with block, with `value` bound to the as-name
    ...
except BaseException:
    if not manager.__exit__(*sys.exc_info()):
        raise
else:
    manager.__exit__(None, None, None)

Two points matter. First, `__enter__` returns the value bound by `as`; it is often `self` but does not have to be. Second, `__exit__` receives the exception that propagated out of the block, or three `None` values if the block exited normally. Returning a truthy value from `__exit__` suppresses the exception; returning a falsy value (including `None`) lets it continue propagating after cleanup.

## 4. Writing a context manager as a class

A frequent tm1py case is a writer that buffers cell updates and flushes them as a single batch on exit:

In [ ]:
from types import TracebackType
from TM1py import TM1Service

class SalesPlanWriter:
    def __init__(self, tm1: TM1Service, cube: str = "Sales Plan") -> None:
        self.tm1 = tm1
        self.cube = cube
        self.pending: dict[tuple[str, ...], float] = {}

    def set(self, key: tuple[str, ...], value: float) -> None:
        self.pending[key] = value

    def __enter__(self) -> "SalesPlanWriter":
        self.pending.clear()
        return self

    def __exit__(
        self,
        exc_type: type[BaseException] | None,
        exc_value: BaseException | None,
        tb: TracebackType | None,
    ) -> None:
        if exc_type is not None:
            self.pending.clear()         # discard the buffer on error
            return
        if self.pending:
            self.tm1.cells.write_values(self.cube, self.pending)
            self.pending.clear()

Used together:

In [ ]:
with TM1Service(...) as tm1, SalesPlanWriter(tm1) as writer:
    writer.set(("2026", "Jan", "Europe", "Revenue"), 120_000.0)
    writer.set(("2026", "Feb", "Europe", "Revenue"), 135_000.0)
# writes are flushed here, but only if the block did not raise

The writer encapsulates a real invariant: a partial flush on error would leave the cube half updated. Pushing that decision into `__exit__` makes it impossible to forget at the call site.

## 5. Returning a value from `__enter__`

The object after `as` is whatever `__enter__` returns. It is often the manager itself, but it need not be. `open(path)` returns a file object, and a database connection's `with` block typically yields a cursor rather than the connection. The same idea applies to a tm1py session that exposes a narrower facade for the duration of the block:

In [ ]:
class SalesPlanSession:
    def __init__(self, tm1: TM1Service) -> None:
        self.tm1 = tm1

    def __enter__(self) -> "SalesPlanCursor":
        return SalesPlanCursor(self.tm1, cube="Sales Plan")

    def __exit__(self, *exc: object) -> None:
        return None

This separation lets the manager hold lifecycle state while the `as`-bound value exposes only the operations the block actually needs. It is also why `__enter__` is the right place for setup that produces a value, rather than `__init__`: construction can happen long before the block runs, and only the `with` should pay the cost of acquiring the resource.

## 6. Handling exceptions in `__exit__`

`__exit__` is the only place where a context manager sees the exception. The three arguments give the exception type, the exception instance, and the traceback. The return value decides what happens next:

In [ ]:
class SuppressNotFound:
    def __enter__(self) -> "SuppressNotFound":
        return self

    def __exit__(self, exc_type, exc_value, tb) -> bool:
        from TM1py.Exceptions import TM1pyRestException
        if exc_type is None:
            return False
        if issubclass(exc_type, TM1pyRestException) and exc_value.status_code == 404:
            return True   # swallow "not found"
        return False      # re-raise everything else

Returning `True` from `__exit__` is a deliberate and quiet choice: callers will not see the exception. Use it sparingly. Logging or annotating the failure inside `__exit__` and then returning `False` is usually a better fit. `__exit__` should not itself raise except to replace the exception with a more meaningful one; raising a new exception there discards the original unless it is explicitly chained with `raise NewError(...) from exc_value`.

## 7. The `contextlib.contextmanager` decorator

Writing a class for every context manager is verbose when the only state is "do this, yield, then do that". `contextlib.contextmanager` lets a single generator stand in for `__enter__` and `__exit__`:

In [ ]:
from contextlib import contextmanager
from time import perf_counter
from typing import Iterator

@contextmanager
def timed(label: str) -> Iterator[None]:
    start = perf_counter()
    try:
        yield
    finally:
        elapsed = perf_counter() - start
        print(f"{label}: {elapsed:.3f}s")

with timed("load Sales Plan view"):
    rows = tm1.cells.execute_view("Sales Plan", "Europe Revenue 2026")
# load Sales Plan view: 0.412s

The single `yield` is the boundary between setup and teardown. Code before the yield runs in `__enter__`; code after runs in `__exit__`. The `try`/`finally` is what gives the same exception guarantee as a class form. To suppress an exception from a generator manager, catch it inside the generator and do not re-raise; to inspect it, catch it, act, then optionally re-raise. A given decorated function returns a fresh, single use manager on each call; the same returned object cannot be used in two `with` blocks (see Topic 12).

## 8. Multiple context managers in one statement

A single `with` can manage several resources, separated by commas. The managers enter left to right and exit right to left:

In [ ]:
with TM1Service(...) as tm1, timed("monthly load"), SalesPlanWriter(tm1) as writer:
    rows = tm1.cells.execute_view("Sales Plan", "Plan Input")
    for key, value in adjusted(rows):
        writer.set(key, value)

This is exactly equivalent to nesting three `with` blocks, but flat. Parentheses around the manager list are allowed since Python 3.10 and let the line wrap cleanly:

In [ ]:
with (
    TM1Service(...) as tm1,
    timed("monthly load"),
    SalesPlanWriter(tm1) as writer,
):
    ...

If `SalesPlanWriter` raises in its `__enter__`, `timed` and `tm1` have already entered and will be exited in reverse order. The protocol guarantees that partially entered groups are torn down correctly.

## 9. `ExitStack` for dynamic resources

Sometimes the count of resources is not known at write time: one connection per region, one cube per scenario, a list determined by configuration. `contextlib.ExitStack` collects context managers at runtime and exits all of them on block exit:

In [ ]:
from contextlib import ExitStack
from TM1py import TM1Service

servers = ["tm1-emea", "tm1-amer", "tm1-apac"]
with ExitStack() as stack:
    services = [
        stack.enter_context(
            TM1Service(address=host, port=8001, user="admin", password="apple", ssl=True)
        )
        for host in servers
    ]
    totals = {
        svc.server.get_server_name(): svc.cubes.get_all_names()
        for svc in services
    }
# every TM1Service is logged out here, in reverse order

`stack.enter_context(cm)` calls `cm.__enter__()` and registers `cm.__exit__` for later. `stack.callback(fn, *args)` registers a plain callable. `ExitStack` is the right tool whenever the number of managers depends on data, when one manager's creation depends on another's result, or when ownership of cleanup must be handed off (`stack.pop_all()` returns a fresh stack the caller can transfer elsewhere).

## 10. Asynchronous context managers

`async with` is the asynchronous counterpart, used inside `async def` functions. The protocol is `__aenter__` and `__aexit__`, both coroutines. tm1py itself is synchronous, but async appears in adjacent code: an async HTTP service that fans out several REST calls, or a Jupyter kernel that mixes async cells with tm1py work.

In [ ]:
import asyncio
import httpx

class TM1AsyncSession:
    def __init__(self, base_url: str, token: str) -> None:
        self.client = httpx.AsyncClient(base_url=base_url, headers={"Authorization": token})

    async def __aenter__(self) -> "TM1AsyncSession":
        return self

    async def __aexit__(self, exc_type, exc_value, tb) -> None:
        await self.client.aclose()

async def main() -> None:
    async with TM1AsyncSession("https://tm1.example.com", token="...") as session:
        responses = await asyncio.gather(
            session.client.get("/api/v1/Cubes('Sales Plan')"),
            session.client.get("/api/v1/Cubes('General Ledger')"),
        )

A synchronous context manager cannot be used with `async with`, and an async one cannot be used with plain `with`. `contextlib.asynccontextmanager` is the async twin of `contextmanager` for generator forms. Reach for async only when the surrounding program is already async; introducing it just to wrap a synchronous tm1py call buys nothing and complicates the call site.

## 11. Real-world design principles

**Reach for a context manager whenever cleanup must run.** If the code wraps resource use in `try`/`finally`, that resource wants to be a context manager. The win is not lines saved; it is that the cleanup obligation moves from every caller to the resource itself.

**Always wrap `TM1Service` in `with`.** A leaked TM1 login holds a server side session until it times out. In long running services or notebooks, leaks accumulate over the day and surface later as confusing "too many sessions" failures.

**Keep `__enter__` cheap; do real work inside the block.** `__enter__` should perform the minimum needed to make the resource usable. Long running setup belongs in a method called from inside the block, where exceptions surface in the obvious place. The `with` line should read like the work, not the plumbing.

**Do not suppress exceptions silently.** A context manager that returns `True` from `__exit__` makes failures invisible. If suppression is the right behavior, name the manager for it (`suppress_not_found`) so the call site advertises the intent.

**Prefer the generator form for one off helpers.** A timer, a temporary working directory, a "switch to this sandbox" helper used in one module: `@contextmanager` is shorter and reads top to bottom. Reach for a class when the manager has methods of its own, when it is reused across modules, or when reentrancy or subclassing matters.

**Use `ExitStack` instead of nested `with` chains.** When the depth of the chain depends on data, nesting becomes brittle and unreadable. `ExitStack` keeps the entries flat and the exits ordered correctly.

## 12. Common mistakes

**Forgetting `with` on `TM1Service`.**

In [ ]:
# Wrong
tm1 = TM1Service(address="tm1.example.com", port=8001, user="admin", password="apple", ssl=True)
cubes = tm1.cubes.get_all_names()
# logout never runs if anything above raises

# Correct
with TM1Service(address="tm1.example.com", port=8001, user="admin", password="apple", ssl=True) as tm1:
    cubes = tm1.cubes.get_all_names()

**Doing setup work in `__init__` instead of `__enter__`.**

In [ ]:
# Wrong
class CubeLoader:
    def __init__(self, tm1: TM1Service, cube: str) -> None:
        self.rows = tm1.cells.execute_view(cube, "All")  # work happens at construction
    def __enter__(self): return self
    def __exit__(self, *exc): pass

# Correct
class CubeLoader:
    def __init__(self, tm1: TM1Service, cube: str) -> None:
        self.tm1 = tm1
        self.cube = cube
    def __enter__(self) -> "CubeLoader":
        self.rows = self.tm1.cells.execute_view(self.cube, "All")
        return self
    def __exit__(self, *exc): pass

**Forgetting `try`/`finally` around the `yield` in a generator manager.**

In [ ]:
# Wrong
@contextmanager
def timed(label: str):
    start = perf_counter()
    yield
    print(f"{label}: {perf_counter() - start:.3f}s")  # skipped if the block raises

# Correct
@contextmanager
def timed(label: str):
    start = perf_counter()
    try:
        yield
    finally:
        print(f"{label}: {perf_counter() - start:.3f}s")

**Returning `True` from `__exit__` by accident.**

In [ ]:
# Wrong
def __exit__(self, exc_type, exc_value, tb):
    if exc_type:
        self.cleanup()
    return True   # all exceptions silently swallowed

# Correct
def __exit__(self, exc_type, exc_value, tb):
    if exc_type:
        self.cleanup()
    return False  # let the exception propagate

**Reusing a `contextmanager` generator across multiple `with` blocks.**

In [ ]:
# Wrong
cm = timed("load")
with cm:
    first_load()
with cm:                 # generator is exhausted, raises RuntimeError
    second_load()

# Correct
with timed("load"):
    first_load()
with timed("load"):
    second_load()

**Calling `__enter__` or `__exit__` directly.**

In [ ]:
# Wrong
manager = SalesPlanWriter(tm1)
writer = manager.__enter__()
do_work(writer)
manager.__exit__(None, None, None)   # no exception path; the guarantee is gone

# Correct
with SalesPlanWriter(tm1) as writer:
    do_work(writer)